# Label YouTube Text as either Prediction or Non-Prediction

- **Tasks:**
    1. Load YouTube transcripts extracted from `1-extract_transcripts.ipynb`.
    4. Clean the data.
    5. Create prompt to identify predictions.
    6. Pass each row of text + prompt into the [UF NaviGator Toolkit](https://it.ufl.edu/ai/navigator-toolkit/) or [Groq](https://console.groq.com/home) LLMs.
    7. Perform majority vote (MV) across all LLMs.
    8. Filter by MV label = 1, hence prediction.
    9. Verify that the text is/is not a prediction and update accordingly.

In [1]:
import os
import sys

import re
from typing import List

import pandas as pd
import numpy as np

from tqdm import tqdm
from youtube_transcript_api import YouTubeTranscriptApi

# Get the current working directory of the notebook
notebook_dir = os.getcwd()
# Add the parent directory to the system path
sys.path.append(os.path.join(notebook_dir, '../'))
# from metrics import Metrics
# from feature_extraction import SpacyFeatureExtraction
from data_processing import DataProcessing
from text_generation_models import TextGenerationModelFactory, llm_classify_text

In [2]:
pd.set_option('max_colwidth', 800)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', None)

## Load Data

In [3]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir)
yt_data_path = os.path.join(base_data_path, 'yt', 'whisper_transcripts', 'sports')
transcripts = os.listdir(yt_data_path)

In [4]:
transcripts

['-rjnvL9LL3U.csv',
 '6STv2GFNB6I.csv',
 'AoE8KFAXHSc.csv',
 'FPl-F2k_KtM.csv',
 'fUmJAtFEGn8.csv',
 'jCe-bY1nP7o.csv',
 'LXPQrZV4Cfw.csv',
 'mBK8o5orBbE.csv',
 'MTVAkVkkaz4.csv',
 'Z0xP3GNpjkw.csv',
 'ZZN7BAYeOtc.csv']

Choose the transcript video that you want to pass through the pipeline. 

Adjust the element from "transcripts" list to choose the right "Video ID".

In [5]:
transcript_path = os.path.join(yt_data_path, transcripts[3])

raw_transcripts_df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')

raw_transcripts_df.head()

,Text,Start Time,Duration,Video ID
0,Hello and welcome to the Sharp 600 brought to you by Covers.com and presented by Bet365.,0.00,14.38,FPl-F2k_KtM
1,My name is Jason Logan. I will be your host here for the next 10 minutes as we take our first bites of Super Bowl 60 odds.,14.56,6.28,FPl-F2k_KtM
2,"And joining us for that dinner, for that snack time, is former odds maker, current professional bettor Todd Furman.",21.00,5.28,FPl-F2k_KtM
3,"Todd, happy bye week to you. The Seahawks, the Patriots get a little bit of a breather,",4.80,0.08,FPl-F2k_KtM
4,"a little bit of downtime this week, but I always found the bye was like my busiest time,",9.38,0.12,FPl-F2k_KtM


In [6]:
# dfs = []

# for transcript in tqdm(transcripts):
#     transcript_path = os.path.join(yt_data_path, transcript)
#     df = DataProcessing.load_from_file(path=transcript_path, file_type='csv')
#     dfs.append(df)

# raw_transcripts_df = DataProcessing.concat_dfs(dfs)
# raw_transcripts_df.head()

In [7]:
def clean_data(df) -> list[str]:
    """
    Split running text into sentence-like segments.
    Rules:
    - Split only where a period (.) or question mark (?) is immediately followed by
    whitespace. This avoids splitting decimals like ``4.5`` (digit after the dot,
    not a space).
    - Leading/trailing whitespace is stripped from each segment.
    - Empty segments are dropped.
    Caveat:
    - Abbreviations such as ``Mr. Smith`` still match ". " and may split incorrectly;
    handle those with a tokenizer or an allowlist if needed.
    Parameters
    ----------
    text : dataframe
        Full text to segment (e.g. one block of joined transcript lines).
    Returns
    -------
    list of str
        Non-empty strings, each intended as one sentence or clause.
    """

    text_joined = ' '.join(df.Text.to_list())
    raw_parts = re.split(r'(?<=[.?])\s+', text_joined)
    text_split = [p.strip() for p in raw_parts if p.strip()]

    return text_split

In [8]:
cleaned_transcripts = clean_data(raw_transcripts_df)
cleaned_transcripts

['Hello and welcome to the Sharp 600 brought to you by Covers.com and presented by Bet365.',
 'My name is Jason Logan.',
 'I will be your host here for the next 10 minutes as we take our first bites of Super Bowl 60 odds.',
 'And joining us for that dinner, for that snack time, is former odds maker, current professional bettor Todd Furman.',
 'Todd, happy bye week to you.',
 'The Seahawks, the Patriots get a little bit of a breather, a little bit of downtime this week, but I always found the bye was like my busiest time, at least in terms of what I need to do, the capping, the content, the planning, all these things, everything that we do at Covers for Super Bowl was very, very busy.',
 'For you as a pro better, what do these two weeks look like?',
 'The schedule has changed considerably from what it is during the bye week now versus what it would have been during the bye week 10 years ago.',
 "I mean, J-Lo, we had player props out at most of the prominent books, bet three, six, five, 

In [9]:
len(cleaned_transcripts)

198

## Prompt LLM

### Create Prompt

In [10]:
prompt = """
Role:
    You are a text analyst.

Task:
    Determine whether the input text contains a prediction, projection, or forecast.

Definitions:
    - A prediction is a statement that asserts or implies a future outcome, result, or event.
    - It may include confidence, uncertainty, or probability.
    - It must go beyond describing current or past facts.

Instructions:
    1. Read the text carefully.
    2. Decide if a prediction/projection/forecast is present.
    3. Assign:
        - 1 if a prediction is present
        - 0 if no prediction is present

Output format (strict JSON):
    {
        "label": 0 or 1
    }
"""

### Initialize LLMs

In [11]:
tgmf = TextGenerationModelFactory()

# Option 3: All NaviGator models
# models = tgmf.create_instances(tgmf.get_navigator_model_names())
models = tgmf.create_instances(['llama-3.1-8b-instant', 'llama-3.3-70b-versatile', 'openai/gpt-oss-120b'])
models

{'llama-3.1-8b-instant': <text_generation_models.LlamaInstantTextGenerationModel at 0x1f9dddbb450>,
 'llama-3.3-70b-versatile': <text_generation_models.LlamaVersatileTextGenerationModel at 0x1f9dddbb3d0>,
 'openai/gpt-oss-120b': <text_generation_models.GptOss120bTextGenerationModel at 0x1f9dde7cd90>}

### Pass data + prompt into each LLM

In [12]:
results = []
PROMPT_TYPE = "zero-shot"   # change to "chain-of-thought" when needed

for cleaned_transcript_idx in range(len(cleaned_transcripts)):
    text = cleaned_transcripts[cleaned_transcript_idx]
    print(f"{cleaned_transcript_idx} --- Sentence: {text}")

    for model_name, model in models.items():
        raw_response, llm_label, llm_reasoning = llm_classify_text(
            data=text,
            base_prompt=prompt,
            prompt_type=PROMPT_TYPE,
            model=model
        )

        result = (
            text,
            raw_response,
            llm_label,
            llm_reasoning,
            model_name
        )
        results.append(result)

        if cleaned_transcript_idx < 3:
            if PROMPT_TYPE == "chain-of-thought":
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label} "
                    f"| Reasoning: {llm_reasoning}"
                )
            else:
                print(
                    f" Model: {model_name} "
                    f"| Label: {llm_label}"
                )

0 --- Sentence: Hello and welcome to the Sharp 600 brought to you by Covers.com and presented by Bet365.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: openai/gpt-oss-120b | Label: 0
1 --- Sentence: My name is Jason Logan.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: openai/gpt-oss-120b | Label: 0
2 --- Sentence: I will be your host here for the next 10 minutes as we take our first bites of Super Bowl 60 odds.
 Model: llama-3.1-8b-instant | Label: 0
 Model: llama-3.3-70b-versatile | Label: 0
 Model: openai/gpt-oss-120b | Label: 0
3 --- Sentence: And joining us for that dinner, for that snack time, is former odds maker, current professional bettor Todd Furman.
4 --- Sentence: Todd, happy bye week to you.
5 --- Sentence: The Seahawks, the Patriots get a little bit of a breather, a little bit of downtime this week, but I always found the bye was like my busiest time, at least in terms of what I need 

> - DataFrame below will have the same sentence (in text column) for all three llms (in the llm_name), then next set of rows will be another sentences with same llms.
> - llm_label column: {0: non-prediction, 1: prediction}

In [13]:
results_with_llm_label_df = pd.DataFrame(
    results,
    columns=['text', 'raw_response', 'llm_label', 'llm_reasoning', 'llm_name']
)

# Ensure labels are valid integers (0/1), fallback to -1 if malformed
results_with_llm_label_df['llm_label'] = (
    pd.to_numeric(results_with_llm_label_df['llm_label'], errors='coerce')
      .fillna(-1)
      .astype(int)
)

# Normalize reasoning: keep only when present (chain-of-thought), else None
results_with_llm_label_df['llm_reasoning'] = (
    results_with_llm_label_df['llm_reasoning'].where(
        results_with_llm_label_df['llm_reasoning'].notna(),
        None
    )
)

results_with_llm_label_df.tail(6)

,text,raw_response,llm_label,llm_reasoning,llm_name
588,"We're going to talk, go even deeper on Super Bowl 60 odds, our best bets, our touchdown picks, all coming next Wednesday.","{""label"": 0}",0,None,llama-3.1-8b-instant
589,"We're going to talk, go even deeper on Super Bowl 60 odds, our best bets, our touchdown picks, all coming next Wednesday.","{""label"": 1}",1,None,llama-3.3-70b-versatile
590,"We're going to talk, go even deeper on Super Bowl 60 odds, our best bets, our touchdown picks, all coming next Wednesday.","{\n ""label"": 0\n}",0,None,openai/gpt-oss-120b
591,"And until then, I guess, enjoy whatever you're wagering on this weekend and best of luck with those bets.","{""label"": 0}",0,None,llama-3.1-8b-instant
592,"And until then, I guess, enjoy whatever you're wagering on this weekend and best of luck with those bets.","{""label"": 0}",0,None,llama-3.3-70b-versatile
593,"And until then, I guess, enjoy whatever you're wagering on this weekend and best of luck with those bets.","{""label"": 0}",0,None,openai/gpt-oss-120b


In [14]:
# Reshape to have llm names as column names
results_df = (
    results_with_llm_label_df
    .pivot_table(
        index='text',
        columns='llm_name',
        values='llm_label',
        aggfunc='first'   # one label per model per sentence
    )
    .reset_index()
    .rename(columns={'text': 'Base Sentence'})
)
results_df

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b
0,11 carries the rest of the game and didn't show that big explosive potential even against the much maligned Rams run defense.,0,0,0
1,A big thank you to Todd for joining us once again.,0,0,0
2,A big thanks to Chris behind the scenes.,0,0,0
3,A clear weather game for the Patriots.,0,0,1
4,A couple of drops.,0,0,0
...,...,...,...,...
191,but I would not be running to bet JSN under his receiving total until the 23rd hour let's put it that way all right well I've got a don't here and this is more of an etiquette thing if you're at a Super Bowl party if you're with a group and that group is maybe not of the betting guild don't constantly talk about your bets I'm someone that does this for for a job and I can't stand those people if you listen if you're with your buddies and it's the group ride parlay and this is your group chat that you're with you know party up go crazy have not go nuts but for god's sakes read the room and don't get down on everyone because you're losing some wager celebrate your wins you hit a 25 to 1 first touchdown winner celebrate it have fun if you lose on heads don't throw a fit don't bring everyo...,0,0,0
192,decided he was going to dance around.,0,0,1
193,look we've seen a Patriots running back get snubbed in the past that had 37 catches and like 900 receiving yards this is a chance for the world to right all of its wrongs and give Ramondre the hardware if he has two touchdowns and 75 all-purpose yards give me Ramondre Stevenson 28 to 1 all right I'll see your skill player and raise you a defensive player go on Demarcus Lawrence it only makes sense that a cowboy is going to get to the Super Bowl and show it almost last year I did uh but I'm happy for Tank but let's talk about this defense this is the reason why San Seattle is here This is the reason why they're favored.,0,1,1
194,since he came over at the trade deadline.,0,0,0


### Majority Vote (across LLMs)

> - llm_label column: {0: non-prediction, 1: prediction}

In [15]:
results_df['Majority Vote Label'] = (
    results_df
    .iloc[:, 1:]               # exclude sentence column
    .mode(axis=1)[0]
    .astype(int)
)

results_df.head()

llm_name,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label
0,11 carries the rest of the game and didn't show that big explosive potential even against the much maligned Rams run defense.,0,0,0,0
1,A big thank you to Todd for joining us once again.,0,0,0,0
2,A big thanks to Chris behind the scenes.,0,0,0,0
3,A clear weather game for the Patriots.,0,0,1,0
4,A couple of drops.,0,0,0,0


Add 'Human Annotation', 'Human Reasoning' columns to write human input, and 'Video ID' column to grab the video source from the respective sentence.

In [16]:
results_df[['Human Annotation', 'Human Reasoning']] = np.nan
final_results_df = pd.concat([results_df, raw_transcripts_df[['Video ID']]], axis=1)
final_results_df.head()

,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label,Human Annotation,Human Reasoning,Video ID
0,11 carries the rest of the game and didn't show that big explosive potential even against the much maligned Rams run defense.,0.0,0.0,0.0,0.0,NaN,NaN,FPl-F2k_KtM
1,A big thank you to Todd for joining us once again.,0.0,0.0,0.0,0.0,NaN,NaN,FPl-F2k_KtM
2,A big thanks to Chris behind the scenes.,0.0,0.0,0.0,0.0,NaN,NaN,FPl-F2k_KtM
3,A clear weather game for the Patriots.,0.0,0.0,1.0,0.0,NaN,NaN,FPl-F2k_KtM
4,A couple of drops.,0.0,0.0,0.0,0.0,NaN,NaN,FPl-F2k_KtM


In [17]:
predictions = final_results_df[final_results_df['Majority Vote Label']==1]

In [18]:
predictions

,Base Sentence,llama-3.1-8b-instant,llama-3.3-70b-versatile,openai/gpt-oss-120b,Majority Vote Label,Human Annotation,Human Reasoning,Video ID
18,"And J-Lo, I'm not quite sure the betting market is going to allow me an opportunity to get involved with the number that I've targeted.",1.0,0.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
20,"And even without Zach Charbonnet, I just don't see Kenneth Walker getting to mid-70s or higher.",1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
23,"And one more thing, Belichick, up for Hall of Fame, doesn't get in on the first time around.",1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
41,"But you also have a Seahawks offense that's going to be stepping up in class given the last three games, two against the 49ers, one against the Rams.",1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
45,Do make a case to bet a lot of player performances to come in under their posted totals.,1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
47,Early bets here for Super Bowl 60.,1.0,1.0,0.0,1.0,NaN,NaN,FPl-F2k_KtM
51,"For me, I'm going to go to that Seattle Seahawks backfield though and take Kenneth Walker under his rushing total at 75 and a half.",1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
55,He could put this one away in one catch.,0.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
63,He's not going to have time for downfield plays to develop.,1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM
65,"Hunter Henry over 36.5 yards, receiving minus 110.",1.0,1.0,1.0,1.0,NaN,NaN,FPl-F2k_KtM


In [19]:
base_data_path = DataProcessing.load_base_data_path(notebook_dir=notebook_dir)
save_data_path = os.path.join(base_data_path, "yt", "majority_vote", "sports")
DataProcessing.save_to_file(predictions, path=save_data_path, prefix='batch_11', save_file_type='csv', include_version=False)

Saving CSV file to: c:\Users\adria\OneDrive\Área de Trabalho\UF Data Studio\predictions\prediction_acquition-youtube\../data\yt\majority_vote\sports\batch_11.csv
